Import Required Libraries

In [1]:
import torch
import torch.nn as nn
import time

set important variables

In [2]:
torch.manual_seed(42)

batch_size = 4
sequence_length = 8
embedding_dim = 32
hidden_dim = 64

# [batch, sequence, embedding]
x = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim
)

print("Input:", x.shape)

Input: torch.Size([4, 8, 32])


Implement RNN

In [3]:
Wx = nn.Parameter(
    torch.randn(hidden_dim, embedding_dim) * 0.1
)

Wh = nn.Parameter(
    torch.randn(hidden_dim, hidden_dim) * 0.1
)

b = nn.Parameter(
    torch.zeros(hidden_dim)
)

h = torch.zeros(
    batch_size,
    hidden_dim
)

hidden_states = []

for t in range(sequence_length):

    # Take one timestep
    x_t = x[:, t, :]

    # RNN equation
    h = torch.tanh(
        x_t @ Wx.T +
        h @ Wh.T +
        b
    )

    hidden_states.append(h)

hidden_states = torch.stack(
    hidden_states,
    dim=1
)

print("RNN output:", hidden_states.shape)

RNN output: torch.Size([4, 8, 64])


Implement Self-Attention

In [4]:
W_Q = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
)

W_K = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
)

W_V = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
)

Q = W_Q(x)
K = W_K(x)
V = W_V(x)

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

Q: torch.Size([4, 8, 32])
K: torch.Size([4, 8, 32])
V: torch.Size([4, 8, 32])


Calculate Attention scores

In [5]:
scores = Q @ K.transpose(-2, -1)

print("Scores:", scores.shape)

Scores: torch.Size([4, 8, 8])


Scale and softmax

In [6]:
attention_scores = scores / (
    embedding_dim ** 0.5
)

attention_weights = torch.softmax(
    attention_scores,
    dim=-1
)

print("Attention weights:", attention_weights.shape)

Attention weights: torch.Size([4, 8, 8])


Attention output

In [7]:
attention_output = attention_weights @ V

print("Attention output:", attention_output.shape)

Attention output: torch.Size([4, 8, 32])


Actual Difference with Parallelization

In [8]:
for t in range(sequence_length):

    x_t = x[:, t, :]

    h = torch.tanh(
        x_t @ Wx.T +
        h @ Wh.T +
        b
    )

Self Attention

In [10]:
class SelfAttention(nn.Module):

    def __init__(self, embedding_dim):

        super().__init__()

        self.W_Q = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False
        )

        self.W_K = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False
        )

        self.W_V = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False
        )

        self.scale = embedding_dim ** 0.5

    def forward(self, x):

        # x:
        # [batch, sequence, embedding]

        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # [batch, sequence, sequence]

        scores = Q @ K.transpose(-2, -1)

        scores = scores / self.scale

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # [batch, sequence, embedding]

        output = attention_weights @ V

        return output, attention_weights

self_attention = SelfAttention(embedding_dim)

output, weights = self_attention(x)

print("Output:", output.shape)
print("Weights:", weights.shape)

# Run this
output, weights = self_attention(x)

print(output.shape)
print(weights.shape)

Output: torch.Size([4, 8, 32])
Weights: torch.Size([4, 8, 8])
torch.Size([4, 8, 32])
torch.Size([4, 8, 8])
